In [2]:
import pandas as pd
import earthaccess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

# 登录 NASA Earthdata
auth = earthaccess.login()

# 配置
CSV_PATH = "/data2/yuyao/methane_emission/preprocess_dataset_L89/merged_with_emit_tag.csv"
EMIT_RAW_DIR = Path("/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/raw_data_dir_EMIT")
EMIT_RAW_DIR.mkdir(exist_ok=True)

# 1. 筛选数据
df = pd.read_csv(CSV_PATH)
df['datetime'] = pd.to_datetime(df['datetime'])

# 筛选条件：Permian Basin 范围 + 时间 < 2024-12-31
# Permian 典型范围: Lat [30, 34], Lon [-105, -101]
mask = (
    (df['plume_latitude'] >= 30) & (df['plume_latitude'] <= 34) &
    (df['plume_longitude'] >= -105) & (df['plume_longitude'] <= -101) &
    (df['datetime'] <= "2024-12-31") &
    (df['has_emit'] == 1)
)
filtered_df = df[mask].drop_duplicates(subset=['emit_granule_id'])

print(f"找到待下载的独特 EMIT 颗粒数量: {len(filtered_df)}")

def download_granule(granule_id):
    """修复后的单任务下载函数"""
    # 检查本地是否已存在，避免重复下载
    if list(EMIT_RAW_DIR.glob(f"*{granule_id}*.nc")):
        return f"{granule_id} 已存在"
    
    # 关键点：必须指定 short_name 或 collection_concept_id
    results = earthaccess.search_data(
        short_name='EMITL2ARFL',  # 显式限定为 EMIT L2A 反射率产品
        granule_name=granule_id,
        count=1
    )
    
    if results:
        earthaccess.download(results, str(EMIT_RAW_DIR))
        return f"{granule_id} 下载完成"
    return f"{granule_id} 未找到"

# 使用线程池加速下载
with ThreadPoolExecutor(max_workers=8) as executor:
    results = list(executor.map(download_granule, filtered_df['emit_granule_id']))

找到待下载的独特 EMIT 颗粒数量: 79


QUEUEING TASKS | : 100%|██████████| 3/3 [00:00<00:00, 2362.99it/s]
QUEUEING TASKS | : 100%|██████████| 3/3 [00:00<00:00, 391.21it/s]

QUEUEING TASKS | : 100%|██████████| 3/3 [00:00<00:00, 1185.50it/s]


QUEUEING TASKS | : 100%|██████████| 3/3 [00:00<00:00, 2693.84it/s]





QUEUEING TASKS | : 100%|██████████| 3/3 [00:00<00:00, 2541.49it/s]







QUEUEING TASKS | : 100%|██████████| 3/3 [00:00<00:00, 2866.27it/s]















QUEUEING TASKS | : 100%|██████████| 3/3 [00:00<00:00, 789.24it/s]





QUEUEING TASKS | : 100%|██████████| 3/3 [00:00<00:00, 229.70it/s]



























PROCESSING TASKS | :  33%|███▎      | 1/3 [01:22<02:45, 82.73s/it]



PROCESSING TASKS | :  67%|██████▋   | 2/3 [04:56<02:39, 159.61s/it]








PROCESSING TASKS | : 100%|██████████| 3/3 [05:09<00:00, 103.31s/it]




COLLECTING RESULTS | : 100%|██████████| 3/3 [00:00<00:00, 65879.12it/s]




QUEUEING TASKS | : 100%|██████████| 3/3 [00:00<00:00, 2232.60it/s]








PROCESSING TASKS | : 100%|██████████

In [4]:
import os
import pandas as pd
import numpy as np
import xarray as xr
import rioxarray
import earthaccess
import time
from pathlib import Path
from pyproj import Transformer, CRS
from scipy.interpolate import interp1d
import cupy as cp
from scipy.spatial import KDTree as CPU_KDTree
from datetime import datetime

# ==================== 用户配置区 ====================
CSV_PATH = "./merged_with_emit_tag.csv"
SRF_CSV = "./landsat9_oli_srf.csv"
EMIT_RAW_DIR = Path("/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/raw_data_dir_EMIT")
OUTPUT_DIR = Path("/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/raw_data_dir_l89_L2SR/EMIT_simulated_landsat9")

# 空间参数
CHIP_SIZE_PX = 512    # 输出 512x512 像素
SCALE_M = 30          # 模拟 Landsat 的 30米分辨率
# ===================================================

EMIT_RAW_DIR.mkdir(exist_ok=True, parents=True)
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# 登录 NASA Earthdata
auth = earthaccess.login()

def get_utm_crs(lat, lon):
    """根据经纬度自动计算合适的 UTM 投影 EPSG 代码"""
    zone = int((lon + 180) / 6) + 1
    if lat >= 0:
        epsg = f"EPSG:326{zone:02d}"
    else:
        epsg = f"EPSG:327{zone:02d}"
    return epsg

def download_emit_granule(granule_id):
    """下载 EMIT 数据"""
    # 检查本地是否已有 RFL 文件
    existing = list(EMIT_RAW_DIR.glob(f"*{granule_id}*RFL*.nc"))
    if existing:
        return existing[0]
    
    results = earthaccess.search_data(short_name='EMITL2ARFL', granule_name=granule_id)
    if not results:
        return None
    
    # 过滤掉不必要的文件，只留 RFL
    links = [link for link in results[0].data_links() if "UNCERT" not in link and "RFL" in link]
    files = earthaccess.download(links, str(EMIT_RAW_DIR))
    return Path(files[0]) if files else None

def process_emit_to_simulated_landsat(row, srf_df):
    plume_id = row['plume_id']
    granule_id = row['emit_granule_id']
    lat, lon = row['plume_latitude'], row['plume_longitude']
    
    out_tif = OUTPUT_DIR / f"{plume_id}_sim_L9.tif"
    if out_tif.exists():
        print(f"   [Skip] {plume_id} exists.")
        return

    print(f"\n[Processing] {plume_id} | Granule: {granule_id}")

    # 1. 下载/加载 EMIT 数据
    rfl_path = download_emit_granule(granule_id)
    if not rfl_path:
        print(f"   [Error] Could not find/download EMIT for {plume_id}")
        return

    # 2. 读取 EMIT 光谱与坐标
    ds = xr.open_dataset(rfl_path, engine='netcdf4')
    ds_band = xr.open_dataset(rfl_path, group='sensor_band_parameters', engine='netcdf4')
    ds_loc = xr.open_dataset(rfl_path, group='location', engine='netcdf4')
    
    waves = ds_band['wavelengths'].values
    e_lon, e_lat = ds_loc['lon'].values, ds_loc['lat'].values
    rfl_val = ds['reflectance'].values # (reflectance_y, reflectance_x, bands)
    
    # 3. 创建目标网格 (Reference Grid)
    # 计算 UTM 坐标以生成以米为单位的正方形
    utm_epsg = get_utm_crs(lat, lon)
    transformer_to_utm = Transformer.from_crs("EPSG:4326", utm_epsg, always_xy=True)
    center_x, center_y = transformer_to_utm.transform(lon, lat)
    
    half_size = (CHIP_SIZE_PX * SCALE_M) / 2
    # 生成标准的网格坐标轴
    target_x = np.linspace(center_x - half_size, center_x + half_size, CHIP_SIZE_PX)
    target_y = np.linspace(center_y + half_size, center_y - half_size, CHIP_SIZE_PX) # Y通常递减
    
    # 将目标网格点转回经纬度，用于在 EMIT 原始数据中索引
    transformer_to_wgs84 = Transformer.from_crs(utm_epsg, "EPSG:4326", always_xy=True)
    mesh_x, mesh_y = np.meshgrid(target_x, target_y)
    t_lon, t_lat = transformer_to_wgs84.transform(mesh_x, mesh_y)

    # 4. GPU 光谱卷积 (EMIT -> Landsat Bands)
    # EMIT shape: (Y, X, 285)
    cp_rfl = cp.array(np.nan_to_num(rfl_val, 0))
    sim_7band_list = []
    for i in range(1, 8):
        f = interp1d(srf_df["wavelength"], srf_df[f"b{i}"], fill_value=0, bounds_error=False)
        w_weights = cp.array(f(waves))
        w_weights /= (w_weights.sum() + 1e-12)
        # 对最后一个维度进行卷积
        sim_7band_list.append(cp.tensordot(cp_rfl, w_weights, axes=(2, 0)))
    
    cp_sim_stack = cp.stack(sim_7band_list) # (7, Y_emit, X_emit)

    # 5. 空间重采样 (使用 CPU 计算索引，GPU 提取数据)
    # 将 EMIT 的经纬度展平
    e_lat_flat = e_lat.ravel()
    e_lon_flat = e_lon.ravel()
    e_coords = np.stack([e_lat_flat, e_lon_flat], axis=1)

    # 将目标网格经纬度展平
    t_coords = np.stack([t_lat.ravel(), t_lon.ravel()], axis=1)

    # 在 CPU 上构建树并查询（对 512x512 的量级，通常只需 < 1秒）
    tree = CPU_KDTree(e_coords)
    max_dist = 0.001  # 约 100 米的经纬度距离，根据实际情况调整
    dist, indices = tree.query(t_coords, distance_upper_bound=max_dist)

    # 处理无效索引 (KDTree 找不到时会返回 len(e_coords))
    invalid_mask = dist == float('inf')
    # 将索引转为 GPU 数组，以便在显存中提取像素
    cp_indices = cp.array(indices)
    cp_invalid_mask = cp.array(invalid_mask).reshape((CHIP_SIZE_PX, CHIP_SIZE_PX))

    final_output = cp.zeros((7, CHIP_SIZE_PX, CHIP_SIZE_PX), dtype=cp.float32)
    # 在循环中应用掩膜
    for b in range(7):
        # 在 GPU 上进行大规模数据重组
        band_flat = cp_sim_stack[b].ravel()
        extracted = band_flat[cp_indices].reshape((CHIP_SIZE_PX, CHIP_SIZE_PX))
        # 将超出范围的点设为 0 或 NaN
        extracted[cp_invalid_mask] = 0 
        final_output[b] = extracted

    # 6. 保存为 GeoTIFF
    sim_da = xr.DataArray(
        final_output.get(),
        dims=("band", "y", "x"),
        coords={"band": np.arange(1, 8), "y": target_y, "x": target_x}
    )
    sim_da.rio.write_crs(utm_epsg, inplace=True)
    sim_da.rio.to_raster(out_tif)
    print(f"   [Success] Saved to {out_tif}")

def main():
    df = pd.read_csv(CSV_PATH)
    srf_df = pd.read_csv(SRF_CSV)
    
    # 筛选条件（可根据需要调整）
    # 目标区域：Permian Basin 示例
    mask = (df['plume_latitude'] >= 30) & (df['plume_latitude'] <= 35) & \
           (df['has_emit'] == 1)
    target_df = df[mask]
    
    print(f"Total tasks: {len(target_df)}")

    for _, row in target_df.iterrows():
        try:
            process_emit_to_simulated_landsat(row, srf_df)
        except Exception as e:
            print(f"   [Error] Task {row['plume_id']} failed: {e}")

if __name__ == "__main__":
    main()

Total tasks: 1264
   [Skip] GAO20230819t193458p0000-A exists.
   [Skip] GAO20230819t193458p0000-G exists.
   [Skip] GAO20230819t193458p0000-N exists.
   [Skip] GAO20230819t193458p0000-T exists.
   [Skip] GAO20230819t194058p0000-A exists.
   [Skip] GAO20230819t194058p0000-B exists.
   [Skip] GAO20230819t194058p0000-F exists.
   [Skip] GAO20230819t194733p0000-A exists.
   [Skip] GAO20230819t194733p0000-F exists.
   [Skip] GAO20230819t195323p0000-C exists.
   [Skip] GAO20230819t195323p0000-F exists.
   [Skip] GAO20230819t195323p0000-G exists.
   [Skip] GAO20230819t200018p0000-A exists.
   [Skip] GAO20230819t200018p0000-B exists.
   [Skip] GAO20230819t200018p0000-C exists.
   [Skip] GAO20230820t160517p0000-A exists.
   [Skip] GAO20230820t161142p0000-A exists.
   [Skip] GAO20230820t161142p0000-B exists.
   [Skip] GAO20230820t161902p0000-A exists.
   [Skip] GAO20230820t161902p0000-B exists.
   [Skip] GAO20230820t161902p0000-C exists.
   [Skip] GAO20230820t162503p0000-A exists.
   [Skip] GAO2

QUEUEING TASKS | : 100%|██████████| 2/2 [00:00<00:00, 3284.50it/s]
PROCESSING TASKS | : 100%|██████████| 2/2 [00:00<00:00, 49636.73it/s]
COLLECTING RESULTS | : 100%|██████████| 2/2 [00:00<00:00, 79137.81it/s]


   [Success] Saved to /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/raw_data_dir_l89_L2SR/EMIT_simulated_landsat9/GAO20230821t171043p0000-I_sim_L9.tif

[Processing] GAO20230821t171043p0000-U | Granule: EMIT_L2A_RFL_001_20230820T192855_2323213_028


QUEUEING TASKS | : 100%|██████████| 2/2 [00:00<00:00, 3966.24it/s]
PROCESSING TASKS | : 100%|██████████| 2/2 [00:00<00:00, 49932.19it/s]
COLLECTING RESULTS | : 100%|██████████| 2/2 [00:00<00:00, 76959.71it/s]


   [Success] Saved to /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/raw_data_dir_l89_L2SR/EMIT_simulated_landsat9/GAO20230821t171043p0000-U_sim_L9.tif

[Processing] GAO20230821t171043p0000-V | Granule: EMIT_L2A_RFL_001_20230820T192855_2323213_028


...

   [Success] Saved to /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/raw_data_dir_l89_L2SR/EMIT_simulated_landsat9/emi20230419t113323p08008-A_sim_L9.tif

[Processing] emi20230420t104534p07015-A | Granule: EMIT_L2A_RFL_001_20230420T104534_2311007_015


QUEUEING TASKS | : 100%|██████████| 2/2 [00:00<00:00, 2864.96it/s]
PROCESSING TASKS | : 100%|██████████| 2/2 [00:43<00:00, 21.85s/it]
COLLECTING RESULTS | : 100%|██████████| 2/2 [00:00<00:00, 54471.48it/s]


   [Success] Saved to /mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/raw_data_dir_l89_L2SR/EMIT_simulated_landsat9/emi20230420t104534p07015-A_sim_L9.tif

[Processing] emi20230420t182847p12010-C | Granule: EMIT_L2A_RFL_001_20230420T182847_2311012_010


QUEUEING TASKS | : 100%|██████████| 2/2 [00:00<00:00, 3204.20it/s]
PROCESSING TASKS | :  50%|█████     | 1/2 [00:34<00:34, 34.25s/it]


KeyboardInterrupt: 